In [90]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head() #datasetul asta e IMENS

,sample_id,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,delay
0,194048,2010,6,YV,Mesa Airlines Inc.,MEM,"Memphis, TN: Memphis International",258.0,63.0,34.00,2.00,9.57,0.0,17.43,7.0,0.0,3514.0
1,200607,2010,1,MQ,American Eagle Airlines Inc.,SAN,"San Diego, CA: San Diego International",512.0,61.0,17.47,5.49,16.85,0.0,21.19,15.0,0.0,2444.0
2,253640,2007,3,OO,SkyWest Airlines Inc.,BTV,"Burlington, VT: Burlington International",66.0,27.0,11.00,0.00,0.00,0.0,16.00,3.0,0.0,1421.0
3,61936,2019,4,EV,ExpressJet Airlines LLC,PVD,"Providence, RI: Theodore Francis Green State",89.0,29.0,9.84,0.00,3.74,0.0,15.42,5.0,0.0,2376.0
4,220886,2008,11,DL,Delta Air Lines Inc.,SDF,"Louisville, KY: Louisville Muhammad Ali Intern...",82.0,24.0,7.79,0.77,6.08,0.0,9.36,0.0,1.0,1505.0


In [91]:
df_train.dtypes

sample_id             int64
year                  int64
month                 int64
carrier              object
carrier_name         object
airport              object
airport_name         object
arr_flights         float64
arr_del15           float64
carrier_ct          float64
weather_ct          float64
nas_ct              float64
security_ct         float64
late_aircraft_ct    float64
arr_cancelled       float64
arr_diverted        float64
delay               float64
dtype: object

In [92]:
df_train.isna().sum()

sample_id             0
year                  0
month                 0
carrier               4
carrier_name          4
airport               2
airport_name          0
arr_flights         397
arr_del15           576
carrier_ct          394
weather_ct          395
nas_ct              392
security_ct         392
late_aircraft_ct    392
arr_cancelled       392
arr_diverted        393
delay               397
dtype: int64

In [93]:
df_train.columns

Index(['sample_id', 'year', 'month', 'carrier', 'carrier_name', 'airport',
       'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct',
       'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled',
       'arr_diverted', 'delay'],
      dtype='object')

In [96]:
#preprocessing - imputing with median and mode, label encoding ( not really)

to_impute_num = ['arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
to_impute_cat = ['carrier', 'carrier_name', 'airport', 'airport_name']

df_train.delay = df_train.delay.fillna(df_train.delay.mean())
for col in to_impute_num:
    df_train[col] = df_train[col].fillna(df_train[col].median())
    df_test[col] = df_test[col].fillna(df_test[col].median())
for col in to_impute_cat:
    df_train[col] = df_train[col].fillna(df_train[col].mode()[0])
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

to_encode = ['carrier', 'carrier_name', 'airport', 'airport_name']
from sklearn.preprocessing import LabelEncoder
#for col in to_encode:
    #enc = LabelEncoder()
    #enc.fit(pd.concat([df_train[col], df_test[col]]))
    #df_train[col] = enc.transform(df_train[col])
    #df_test[col] = enc.transform(df_test[col])

# transforming some counts into rates
df_train["cancel_rate"] = df_train["arr_cancelled"] / df_train["arr_flights"].clip(lower=1)
df_train["divert_rate"] = df_train["arr_diverted"] / df_train["arr_flights"].clip(lower=1)
df_train["carrier_rate"] = df_train["carrier_ct"] / df_train["arr_del15"].clip(lower=1) # astea cauzeaza intarzieri si d asta luam rate ul din alea intarziate
df_train["weather_rate"] = df_train["weather_ct"] / df_train["arr_del15"].clip(lower=1)

df_test["cancel_rate"] = df_test["arr_cancelled"] / df_test["arr_flights"].clip(lower=1)
df_test["divert_rate"] = df_test["arr_diverted"] / df_test["arr_flights"].clip(lower=1)
df_test["carrier_rate"] = df_test["carrier_ct"] / df_test["arr_del15"].clip(lower=1)
df_test["weather_rate"] = df_test["weather_ct"] / df_test["arr_del15"].clip(lower=1)

df_train.head()

,sample_id,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,...,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,delay,cancel_rate,divert_rate,carrier_rate,weather_rate
0,194048,2010,6,YV,Mesa Airlines Inc.,MEM,"Memphis, TN: Memphis International",258.0,63.0,34.00,...,9.57,0.0,17.43,7.0,0.0,3514.0,0.027132,0.000000,0.539683,0.031746
1,200607,2010,1,MQ,American Eagle Airlines Inc.,SAN,"San Diego, CA: San Diego International",512.0,61.0,17.47,...,16.85,0.0,21.19,15.0,0.0,2444.0,0.029297,0.000000,0.286393,0.090000
2,253640,2007,3,OO,SkyWest Airlines Inc.,BTV,"Burlington, VT: Burlington International",66.0,27.0,11.00,...,0.00,0.0,16.00,3.0,0.0,1421.0,0.045455,0.000000,0.407407,0.000000
3,61936,2019,4,EV,ExpressJet Airlines LLC,PVD,"Providence, RI: Theodore Francis Green State",89.0,29.0,9.84,...,3.74,0.0,15.42,5.0,0.0,2376.0,0.056180,0.000000,0.339310,0.000000
4,220886,2008,11,DL,Delta Air Lines Inc.,SDF,"Louisville, KY: Louisville Muhammad Ali Intern...",82.0,24.0,7.79,...,6.08,0.0,9.36,0.0,1.0,1505.0,0.000000,0.012195,0.324583,0.032083


In [97]:
features = ['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct',
            'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'cancel_rate', 'divert_rate', 'carrier_rate', 'weather_rate']
target = 'delay'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,cancel_rate,divert_rate,carrier_rate,weather_rate
0,2010,6,YV,Mesa Airlines Inc.,MEM,"Memphis, TN: Memphis International",258.0,63.0,34.00,2.00,9.57,0.0,17.43,7.0,0.0,0.027132,0.000000,0.539683,0.031746
1,2010,1,MQ,American Eagle Airlines Inc.,SAN,"San Diego, CA: San Diego International",512.0,61.0,17.47,5.49,16.85,0.0,21.19,15.0,0.0,0.029297,0.000000,0.286393,0.090000
2,2007,3,OO,SkyWest Airlines Inc.,BTV,"Burlington, VT: Burlington International",66.0,27.0,11.00,0.00,0.00,0.0,16.00,3.0,0.0,0.045455,0.000000,0.407407,0.000000
3,2019,4,EV,ExpressJet Airlines LLC,PVD,"Providence, RI: Theodore Francis Green State",89.0,29.0,9.84,0.00,3.74,0.0,15.42,5.0,0.0,0.056180,0.000000,0.339310,0.000000
4,2008,11,DL,Delta Air Lines Inc.,SDF,"Louisville, KY: Louisville Muhammad Ali Intern...",82.0,24.0,7.79,0.77,6.08,0.0,9.36,0.0,1.0,0.000000,0.012195,0.324583,0.032083


In [103]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

cat_features = ['carrier', 'carrier_name', 'airport', 'airport_name']

model = CatBoostRegressor(random_state=42, cat_features=cat_features, iterations=5000, verbose=250)

#cross_val_score(model, X, y, cv=2, scoring='neg_mean_absolute_error')

In [104]:
model.fit(X, y)
y_preds = model.predict(X_final)

Learning rate set to 0.026546
0:	learn: 12212.5256556	total: 172ms	remaining: 14m 19s
250:	learn: 1968.5390445	total: 43s	remaining: 13m 34s
500:	learn: 1664.6216328	total: 1m 18s	remaining: 11m 45s
750:	learn: 1518.1712683	total: 1m 55s	remaining: 10m 55s
1000:	learn: 1401.9543616	total: 2m 34s	remaining: 10m 17s
1250:	learn: 1320.5953853	total: 3m 29s	remaining: 10m 27s
1500:	learn: 1260.2141569	total: 4m 16s	remaining: 9m 57s
1750:	learn: 1208.3154145	total: 5m 5s	remaining: 9m 26s
2000:	learn: 1162.8553366	total: 5m 54s	remaining: 8m 51s
2250:	learn: 1127.3213394	total: 6m 43s	remaining: 8m 13s
2500:	learn: 1095.6734970	total: 7m 33s	remaining: 7m 33s
2750:	learn: 1065.1965369	total: 8m 21s	remaining: 6m 49s
3000:	learn: 1038.7540032	total: 9m 10s	remaining: 6m 7s
3250:	learn: 1014.5843031	total: 9m 59s	remaining: 5m 22s
3500:	learn: 991.9313036	total: 10m 48s	remaining: 4m 37s
3750:	learn: 972.3221838	total: 11m 37s	remaining: 3m 52s
4000:	learn: 952.6100470	total: 12m 25s	remaini

In [105]:
submission = pd.DataFrame({
    "sample_id": df_test.sample_id,
    "delay": y_preds
})

submission

,sample_id,delay
0,184678,775.056707
1,78226,1434.576836
2,124301,5687.152256
3,206506,984.417709
4,133174,1965.625288
...,...,...
63599,168537,5152.542987
63600,207672,2608.439561
63601,125851,180.065075
63602,2649,4484.861132


In [106]:
submission.to_csv("submission.csv", index=False)